In [16]:
#gera modelos de sedimentos com deposição, dobras e falhamentos
#imports
import time
from scipy import ndimage
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import os
import random
import seaborn as sns
import cv2
import sys
import struct
from scipy import interpolate
from scipy.ndimage import gaussian_filter

In [ ]:
#para geração de modelos no colab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
#parametros
n_models=500

In [14]:
#tamanho dos modelos a serem gerados
nx=128
nz=128

#número de pacotes a serem depositados e número de camadas em cada pacote
npac=2
pac_size=3
nc=1+npac*pac_size
cam_min=2
cam_max=int((1.0*nz)/(npac*pac_size))

#eventos geológicos
dobra=False
erode=False
falha=False
diretorio='./Mods1/'
# diretorio='/content/drive/MyDrive/PosDoc/Curso_open/Atividade_DL/Modelos01a/'

# dobra=True
# erode=False
# falha=False
# diretorio='/content/drive/MyDrive/PosDoc/Curso_open/Atividade_DL/Modelos02/'

# dobra=False
# erode=False
# falha=True
# diretorio='/content/drive/MyDrive/PosDoc/Curso_open/Atividade_DL/Modelos03/'


#intervalo de velocidades a serem inseridas
vmin=1550
vmax=3000
wlayer=30
limita_vel=False

#parâmetros dos dobramentos
per_max1=nx/2
per_min1=nx/4
# per_max2=nx/20
# per_min2=nx/50
amp_min1=4
amp_max1=20
# amp_min2=8
# amp_max2=20
fase_min=0
fase_max=int(nx*np.pi)

In [ ]:
for imod in range(1,n_models+1):
    print(imod)
    #inicializações
    nz_pac=nz
    ind_pac=0
    labels=np.zeros(shape=(nz,nx),dtype=int)
    d_max=nz

    #laço sobre o número de pacotes
    for i in range(0,npac):
        ind_pac=ind_pac+1
        if(ind_pac!=1):
            labels[0,:]=0
            labels[nz-1,:]=1
            d_max=np.amin(np.where(labels!=0)[0])
        #deposição das camadas
        for cam in range(0,pac_size):
            # cam_num=nc-(cam+(ind_pac-1)*pac_size)
            cam_num=cam+1+(ind_pac-1)*pac_size
            # print('camn = ',cam_num)
            cam_thick=random.randint(cam_min,cam_max)
            d_max=d_max-cam_thick
            #grava as interfaces
            for x in range(0,nx):
                auxlabels=np.zeros(shape=(nz),dtype=int)
                val=[]
                val=np.where(labels[:,x]==0)
                auxlabels[val]=cam_num
                labels[d_max:nz,x]=labels[d_max:nz,x]+auxlabels[d_max:nz]
        ########################################
        #dobramento das camadas#################
        if (dobra):
          # if(random.randint(0,1) == 1 or i==0):
          periodo=random.uniform(per_min1,per_max1)
          amp=random.uniform(amp_min1,amp_max1)
          # else:
          #     periodo=random.uniform(per_min2,per_max2)
          #     amp=random.uniform(amp_min2,amp_max2)
          fase=random.uniform(fase_min,fase_max)
          labelsaux=np.zeros(shape=(nz,nx),dtype=int)
          # print('amp, periodo', amp,periodo)
          for x in range(0,nx):
              z_s=int((amp*np.sin(x/periodo+fase)))
              for z in range(0,nz):
                  labelsaux[z,x]=labels[np.clip(z+z_s,0,nz-1),x]
          labels=np.copy(labelsaux)
        labels[0,:]=0
        labels[nz-1,:]=1
        zmin=np.amin(np.where(labels!=0)[0])
        #########################################
        #introduz falhas no modelo################
        if (falha):
          escala=nz/nx
          n_falhas=1
          ang=random.uniform(np.pi/12,np.pi/6)
          sinal=random.randint(0,1)
          if sinal ==0:
              ang=(-1)*ang
          alfa=np.tan(ang)*escala
          for ii in range(0,n_falhas):
              rejeito=random.randint(10,30)
              sinal=random.randint(0,1)
              if sinal ==0:
                  rejeito=(-1)*rejeito
              pini=random.randint(50,nx-50)
              labelsaux=np.copy(labels)
              labels[0,:]=0
              labels[nz-1,:]=1
              zref=np.amin(np.where(labels[:,pini]!=0))
              if alfa <= 0:
                  for z in range(0,nz):
                      x_s=int((alfa*(zref-z)))
                      for x in range(np.clip(pini+x_s,0,nx-1),nx):
                          labelsaux[z,x]=labels[np.clip(z+rejeito,0,nz-1),np.clip(x-x_s,0,nx-1)]
                  labels=np.copy(labelsaux)

              if alfa > 0:
                  for z in range(0,nz):
                      x_s=int((alfa*(zref-z)))
                      for x in range(np.clip(pini+x_s,0,nx-1),nx):
                          labelsaux[z,x]=labels[np.clip(z+rejeito,0,nz-1),np.clip(x+x_s,0,nx-1)]
                  labels=np.copy(labelsaux)
        labels[0,:]=0
        labels[nz-1,:]=1
        zmin=np.amin(np.where(labels!=0)[0])
        ################################################
        #erosão- apaga as porções superiores do modelo
        if (erode):
          erosao=random.randint(0,1)
          if erosao == 1:
              perda=random.randint(10,80)
              labels[0:np.clip(zmin+perda,0,nz),:]=0
        #################################################
        labels[0,:]=0
        labels[nz-1,:]=1
        zmin=np.amin(np.where(labels!=0)[0])

    ############# fim do ciclo de deposição###################
    last_label=np.amax(labels)
    #checa se a primeira camada não está muito funda###puxa todas as camadas para cima
    if zmin >= (wlayer+10):
        labels[wlayer+5:wlayer+5+(nz-zmin),:]=labels[zmin:nz,:]
        labels[wlayer+5+(nz-zmin):nz,:]=1#coloca 1 na parte funda
        zmin=np.amin(np.where(labels!=0)[0])

    if zmin <= wlayer:#erode o sedimento até wlayer
        labels[0:wlayer,:]=0

    for x in range(0,nx): #define a última camada até wlayer
        auxlabels=np.zeros(shape=(nz),dtype=int)
        val=[]
        val=np.where(labels[:,x]==0)
        auxlabels[val]=auxlabels[val]+last_label+1
        labels[wlayer:nz,x]=labels[wlayer:nz,x]+auxlabels[wlayer:nz]

    last_label=np.amax(labels)

    #definir as velocidades que irão compor o modelo
    for cam in range(last_label):
        vels=np.linspace(vmin,vmax,last_label+1)
        vels_draw=[]

       
        for i in range(0,len(vels)):
            x=np.random.normal(vels[i],(vels[1]-vels[0]))
            vels_draw.append(np.round(x,2))
    # adiciona camadas intermediárias com alta velocidade
    for i in range(len(vels_draw)):
        if(vels_draw[i] >= vmax):
                vels_draw[i]=vmax
        alta=random.randint(0,9)
        if (alta > 6):
            vels_draw[i]=vels_draw[i]+(random.randint(1,5)*100)
            if(vels_draw[i] >= vmax):
                vels_draw[i]=vmax

    #adiciona camadas intermediárias com baixa velocidade
    for i in range(len(vels_draw)):
        alta=random.randint(0,9)
        if (alta > 6):
            if(vels_draw[i] >= 2800):
                vels_draw[i]=vels_draw[i]-(random.randint(1,5)*100)

    ######gera o modelo de velocidades############
    last_label=np.amax(labels)
    modfinal = np.zeros(shape=(nz,nx))+1500
    maxval=np.amax(vels_draw)
    for z in range(wlayer,nz):
        for x in range(0,nx):
            modfinal[z,x]=vels_draw[int(np.clip(nc-labels[z,x],0,len(vels_draw)-1))]
            if (limita_vel):
              # esses dois if controlam os limites inferiores e superiores de velocidade
              if modfinal[z,x] >= maxval:
                modfinal[z,x]=vmax
              if modfinal[z,x] <= vmin:
                modfinal[z,x]= vmin
    #print(np.amin(modfinal))
    #print(np.amax(modfinal))

    # modfinal.T.tofile(diretorio+'/Bin/model_{}.vp'.format(imod))
    np.save(diretorio+'/Bin/model_{}.vp'.format(imod),modfinal)
    plt.imsave(diretorio+'/Figs/teste_{}.jpeg'.format(imod),modfinal,cmap='jet',vmin=vmin, vmax=vmax)


indmod 1
1500.0
3000.0
indmod 2
1337.52
2835.18
indmod 3
1500.0
2927.04
indmod 4
1157.28
2640.7
indmod 5
1500.0
3000.0
indmod 6
1500.0
2700.84
indmod 7
1500.0
3000.0
indmod 8
1500.0
3000.0
indmod 9
1500.0
3000.0
indmod 10
1500.0
3000.0
indmod 11
1500.0
3000.0
indmod 12
1463.22
2980.84
indmod 13
1500.0
2725.41
indmod 14
1281.53
2780.74
indmod 15
1500.0
3000.0
indmod 16
1500.0
2957.29
indmod 17
1500.0
2869.97
indmod 18
1227.92
2737.24
indmod 19
1389.53
2763.99
indmod 20
1317.83
2912.61
indmod 21
1372.81
2641.65
indmod 22
1500.0
2798.71
indmod 23
1500.0
2944.73
indmod 24
1500.0
2722.81
indmod 25
1466.83
3000.0
indmod 26
1500.0
3000.0
indmod 27
1500.0
2854.63
indmod 28
1405.73
2922.83
indmod 29
1431.24
3000.0
indmod 30
1500.0
3000.0
indmod 31
1500.0
2694.33
indmod 32
1500.0
3000.0
indmod 33
1418.88
2818.45
indmod 34
1500.0
3000.0
indmod 35
1392.01
2670.2
indmod 36
1500.0
2969.63
indmod 37
1500.0
2851.92
indmod 38
1500.0
2636.72
indmod 39
1459.77
2965.47
indmod 40
1500.0
2582.25
indmod 41
1